# Causal-intervention analysis

Reads the per-run artifacts produced by `interp_experiments/run_causal_interventions.py` and visualises:

- **Section 1** sanity stats — conditions per run, baseline values, sample example prompt
- **Section 2** intervention effects on **stated confidence** (steering curves + ablation bars)
- **Section 3** intervention effects on **direct (output) entropy**
- **Section 4** intervention effects on the **Spearman ρ(stated confidence, direct entropy)** — does steering break or preserve the metacognitive coupling?
- **Section 5** intervention effects on **direct accuracy**
- **Section 6** layer × multiplier **heatmaps** for each direction × outcome

Drop one or more run subfolders into `RUN_DIRS` in the next cell and re-run from the top.

In [ ]:
# >>>>>>>>> EDIT THIS LIST <<<<<<<<<
# Paste in one or more run subfolders from outputs/causal_interventions/.
# Order is preserved in the legend; recommended order is base → instruct → finetuned.
RUN_DIRS = [
    'outputs/causal_interventions/8b_instruct_mixed_17173_all_test',
    # 'outputs/causal_interventions/8b_base_mixed_17173_all_test',
    # 'outputs/causal_interventions/8b_20260506-034609_step_300_mixed_17173_all_test',
]
# Optional: override per-run plot labels (None → derive from folder name).
RUN_LABELS = None  # or a list of strings of len(RUN_DIRS)

In [ ]:
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Notebook lives in analysis/ — RUN_DIRS are repo-root-relative.
REPO_ROOT = Path('..').resolve() if Path('analyze_interventions.ipynb').exists() else Path('.').resolve()
print(f'Repo root: {REPO_ROOT}')


def _resolve(p):
    p = Path(p)
    return p if p.is_absolute() else (REPO_ROOT / p).resolve()


def _find_results_json(folder: Path) -> Path:
    """Each run folder has exactly one *_intervention_results.json."""
    candidates = list(folder.glob('*_intervention_results.json'))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected 1 *_intervention_results.json in {folder}, got {len(candidates)}')
    return candidates[0]


def _label_from_folder(folder: Path, dataset_name: str) -> str:
    """folder name = '8b_<tag>_<dataset_short>'; returns <tag>."""
    name = folder.name
    dataset_short = Path(dataset_name).stem if dataset_name.endswith('.jsonl') else dataset_name
    suffix = '_' + dataset_short
    if name.startswith('8b_') and name.endswith(suffix):
        return name[3:-len(suffix)]
    return name


@dataclass
class RunData:
    folder: Path
    label: str
    results: dict
    df: pd.DataFrame  # one row per condition


def _conditions_to_dataframe(results: dict) -> pd.DataFrame:
    """Flatten conditions list to a DataFrame, dropping the heavy per-question arrays."""
    drop_keys = {'stated_confidence_per_question', 'direct_entropy_per_question',
                 'meta_responses', 'direct_responses'}
    rows = []
    for cond in results['conditions']:
        rows.append({k: v for k, v in cond.items() if k not in drop_keys})
    df = pd.DataFrame(rows)
    # Normalize a few columns so plots can rely on them.
    for col in ('direction_type', 'layer', 'multiplier'):
        if col not in df.columns:
            df[col] = np.nan
    if 'kind' not in df.columns:
        df['kind'] = 'unknown'
    return df


def load_run(folder, label=None) -> RunData:
    folder = _resolve(folder)
    results_path = _find_results_json(folder)
    results = json.load(open(results_path))
    if label is None:
        label = _label_from_folder(folder, results.get('dataset_name', folder.name))
    df = _conditions_to_dataframe(results)
    return RunData(folder=folder, label=label, results=results, df=df)


labels = RUN_LABELS if RUN_LABELS else [None] * len(RUN_DIRS)
RUNS: List[RunData] = [load_run(d, l) for d, l in zip(RUN_DIRS, labels)]

# Compact label for a lone finetuned run, mirroring analyze_activations.ipynb.
if not RUN_LABELS:
    finetuned_idx = [i for i, r in enumerate(RUNS) if r.label not in ('base', 'instruct')]
    if len(finetuned_idx) == 1:
        RUNS[finetuned_idx[0]].label = 'finetuned'

if not RUN_LABELS:
    def _order_key(label):
        if label == 'base':      return (0, '')
        if label == 'instruct':  return (1, '')
        if label == 'finetuned': return (2, '')
        return (3, label)
    RUNS.sort(key=lambda r: _order_key(r.label))

print(f'Loaded {len(RUNS)} run(s):')
for r in RUNS:
    res = r.results
    adapter = res.get('model_name', '?')
    base = res.get('base_model_name', '?')
    adapter_str = adapter.split('/')[-1] if adapter != base else '(none)'
    print(f"  {r.label:35s}  model={base}  adapter={adapter_str}")
    print(f"  {'':35s}  dataset={res.get('dataset_name','?')}  n={res.get('num_questions','?')}  scale={res.get('confidence_scale','?')}")
    print(f"  {'':35s}  conditions={len(r.df)}  layers={res.get('intervention_layers')}  mults={res.get('steering_multipliers')}")

---
## Section 1 — Summary & sanity checks

Per-run condition counts, the baseline (no-intervention) values, and one example prompt + response from the saved `_examples.txt`.

In [ ]:
# Per-run baseline + condition summary.
rows = []
for r in RUNS:
    base = r.df[r.df['kind'] == 'baseline']
    n_steer = (r.df['kind'] == 'steer').sum()
    n_ablate = (r.df['kind'] == 'ablate').sum()
    if len(base) == 0:
        rows.append({'run': r.label, 'n_conditions': len(r.df),
                     'n_steer': int(n_steer), 'n_ablate': int(n_ablate),
                     'baseline': '(missing)'})
        continue
    b = base.iloc[0]
    rows.append({
        'run': r.label,
        'n_conditions': len(r.df),
        'n_steer': int(n_steer),
        'n_ablate': int(n_ablate),
        'baseline_H_direct': round(float(b['direct_entropy_mean']), 3),
        'baseline_stated':   round(float(b['stated_confidence_mean']), 3),
        'baseline_soft_stated': round(float(b['soft_stated_confidence_mean']), 3),
        'baseline_rho': round(float(b['spearman_stated_vs_direct_entropy']), 3),
        'baseline_acc': round(float(b['direct_accuracy']), 3),
    })
df_summary = pd.DataFrame(rows)
df_summary

In [ ]:
# Show one example prompt + response per run (from the saved _examples.txt).
for r in RUNS:
    candidates = list(r.folder.glob('*_examples.txt'))
    print('=' * 80)
    print(f'Run: {r.label}  ({r.folder.name})')
    print('=' * 80)
    if not candidates:
        print('  (no _examples.txt found)')
        continue
    txt = candidates[0].read_text()
    # Print the header + first ~40 lines so the notebook stays readable.
    head = '\n'.join(txt.splitlines()[:50])
    print(head)
    print('  …')

---
## Section 2 — Effect on stated confidence

Mean stated confidence (numeric) as a function of steering multiplier, separately for each direction × layer × run. Baseline is the dashed horizontal line. The soft signal (probability-weighted midpoint) is plotted alongside the argmax-decoded value.

If a direction *causally* controls stated confidence, the curve should slope monotonically with the multiplier.

In [ ]:
# Helpers shared across Sections 2-5.
DIRECTION_TYPES_DEFAULT = ['entropy', 'stated_confidence']

def _direction_types(run: RunData) -> List[str]:
    return list(run.results.get('direction_types', DIRECTION_TYPES_DEFAULT))

def _baseline_value(run: RunData, col: str) -> Optional[float]:
    base = run.df[run.df['kind'] == 'baseline']
    if len(base) == 0:
        return None
    return float(base.iloc[0][col])

def _steer_curves(run: RunData, direction_type: str, value_col: str):
    """Return {layer: (mults_array, values_array)} sorted by multiplier."""
    sub = run.df[(run.df['kind'] == 'steer') & (run.df['direction_type'] == direction_type)]
    out = {}
    for layer, g in sub.groupby('layer'):
        g = g.sort_values('multiplier')
        out[int(layer)] = (g['multiplier'].to_numpy(dtype=float), g[value_col].to_numpy(dtype=float))
    return out

def _ablate_values(run: RunData, direction_type: str, value_col: str):
    """Return (layers_array, values_array) for ablation conditions."""
    sub = run.df[(run.df['kind'] == 'ablate') & (run.df['direction_type'] == direction_type)]
    sub = sub.sort_values('layer')
    return sub['layer'].to_numpy(dtype=int), sub[value_col].to_numpy(dtype=float)

def _plot_steering_panel(ax, run: RunData, direction_type: str, value_col: str, *,
                          baseline: Optional[float], ylabel: str):
    curves = _steer_curves(run, direction_type, value_col)
    if not curves:
        ax.text(0.5, 0.5, '(no steer conditions)', ha='center', va='center'); ax.set_axis_off(); return
    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(curves)))
    for c, (layer, (mults, vals)) in zip(cmap, sorted(curves.items())):
        ax.plot(mults, vals, marker='o', color=c, label=f'L{layer}', linewidth=1.5, markersize=4)
    if baseline is not None:
        ax.axhline(baseline, color='k', linestyle='--', linewidth=1, alpha=0.6, label='baseline')
    ax.axvline(0, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('steering multiplier (residual-stream units)')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{run.label} — steer along {direction_type}')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

In [ ]:
# Section 2.1  Stated confidence (argmax-decoded numeric value) vs steering multiplier.
for run in RUNS:
    dts = _direction_types(run)
    fig, axes = plt.subplots(1, len(dts), figsize=(6.5 * len(dts), 4.2), squeeze=False)
    base_val = _baseline_value(run, 'stated_confidence_mean')
    for j, dt in enumerate(dts):
        _plot_steering_panel(axes[0, j], run, dt, 'stated_confidence_mean',
                              baseline=base_val, ylabel='mean stated confidence')
    plt.tight_layout(); plt.show()

In [ ]:
# Section 2.2  Soft stated confidence (probability-weighted midpoint) vs steering multiplier.
for run in RUNS:
    dts = _direction_types(run)
    fig, axes = plt.subplots(1, len(dts), figsize=(6.5 * len(dts), 4.2), squeeze=False)
    base_val = _baseline_value(run, 'soft_stated_confidence_mean')
    for j, dt in enumerate(dts):
        _plot_steering_panel(axes[0, j], run, dt, 'soft_stated_confidence_mean',
                              baseline=base_val, ylabel='mean soft stated confidence')
    plt.tight_layout(); plt.show()

In [ ]:
# Section 2.3  Ablation: change in mean stated confidence vs baseline, per layer.
for run in RUNS:
    dts = _direction_types(run)
    base_val = _baseline_value(run, 'stated_confidence_mean')
    fig, axes = plt.subplots(1, len(dts), figsize=(6.5 * len(dts), 3.8), squeeze=False)
    for j, dt in enumerate(dts):
        ax = axes[0, j]
        layers, vals = _ablate_values(run, dt, 'stated_confidence_mean')
        if len(layers) == 0:
            ax.text(0.5, 0.5, '(no ablate conditions)', ha='center', va='center'); ax.set_axis_off(); continue
        deltas = vals - (base_val if base_val is not None else np.nan)
        bars = ax.bar([str(l) for l in layers], deltas, color='C2', alpha=0.8)
        ax.axhline(0, color='k', linewidth=0.8)
        ax.set_xlabel('intervention layer')
        ax.set_ylabel('Δ stated confidence (vs baseline)')
        ax.set_title(f'{run.label} — ablate {dt}')
        ax.grid(alpha=0.3, axis='y')
        for b, v in zip(bars, deltas):
            ax.text(b.get_x() + b.get_width()/2, v, f'{v:+.2f}', ha='center', va='bottom' if v >= 0 else 'top', fontsize=7)
    plt.tight_layout(); plt.show()

---
## Section 3 — Effect on direct (output) entropy

Same plots as Section 2 but with **mean direct-task entropy** on the y-axis. This is the model's actual uncertainty over A/B/C/D, not its self-report. A direction that controls *internal* uncertainty should move this curve.

In [ ]:
# Section 3.1  Direct entropy vs steering multiplier.
for run in RUNS:
    dts = _direction_types(run)
    fig, axes = plt.subplots(1, len(dts), figsize=(6.5 * len(dts), 4.2), squeeze=False)
    base_val = _baseline_value(run, 'direct_entropy_mean')
    for j, dt in enumerate(dts):
        _plot_steering_panel(axes[0, j], run, dt, 'direct_entropy_mean',
                              baseline=base_val, ylabel='mean direct entropy (nats)')
    plt.tight_layout(); plt.show()

In [ ]:
# Section 3.2  Ablation: Δ direct entropy vs baseline, per layer.
for run in RUNS:
    dts = _direction_types(run)
    base_val = _baseline_value(run, 'direct_entropy_mean')
    fig, axes = plt.subplots(1, len(dts), figsize=(6.5 * len(dts), 3.8), squeeze=False)
    for j, dt in enumerate(dts):
        ax = axes[0, j]
        layers, vals = _ablate_values(run, dt, 'direct_entropy_mean')
        if len(layers) == 0:
            ax.text(0.5, 0.5, '(no ablate conditions)', ha='center', va='center'); ax.set_axis_off(); continue
        deltas = vals - (base_val if base_val is not None else np.nan)
        ax.bar([str(l) for l in layers], deltas, color='C3', alpha=0.8)
        ax.axhline(0, color='k', linewidth=0.8)
        ax.set_xlabel('intervention layer')
        ax.set_ylabel('Δ direct entropy')
        ax.set_title(f'{run.label} — ablate {dt}')
        ax.grid(alpha=0.3, axis='y')
    plt.tight_layout(); plt.show()

---
## Section 4 — Effect on the metacognitive coupling

Spearman ρ between the model's stated confidence and its own output entropy, **per condition**. This is the headline metric the introspection probes were built around — a steered model that moves stated confidence *without* moving output entropy will show ρ collapse, even when the marginals look fine.

Baseline ρ is the dashed horizontal line.

In [ ]:
# Section 4.1  Spearman ρ(stated, direct entropy) vs steering multiplier.
for run in RUNS:
    dts = _direction_types(run)
    fig, axes = plt.subplots(1, len(dts), figsize=(6.5 * len(dts), 4.2), squeeze=False)
    base_val = _baseline_value(run, 'spearman_stated_vs_direct_entropy')
    for j, dt in enumerate(dts):
        _plot_steering_panel(axes[0, j], run, dt, 'spearman_stated_vs_direct_entropy',
                              baseline=base_val, ylabel='Spearman ρ(stated, H_direct)')
    plt.tight_layout(); plt.show()

In [ ]:
# Section 4.2  Ablation: Δ Spearman ρ vs baseline, per layer.
for run in RUNS:
    dts = _direction_types(run)
    base_val = _baseline_value(run, 'spearman_stated_vs_direct_entropy')
    fig, axes = plt.subplots(1, len(dts), figsize=(6.5 * len(dts), 3.8), squeeze=False)
    for j, dt in enumerate(dts):
        ax = axes[0, j]
        layers, vals = _ablate_values(run, dt, 'spearman_stated_vs_direct_entropy')
        if len(layers) == 0:
            ax.text(0.5, 0.5, '(no ablate conditions)', ha='center', va='center'); ax.set_axis_off(); continue
        deltas = vals - (base_val if base_val is not None else np.nan)
        ax.bar([str(l) for l in layers], deltas, color='C4', alpha=0.8)
        ax.axhline(0, color='k', linewidth=0.8)
        ax.set_xlabel('intervention layer')
        ax.set_ylabel('Δ Spearman ρ')
        ax.set_title(f'{run.label} — ablate {dt}')
        ax.grid(alpha=0.3, axis='y')
    plt.tight_layout(); plt.show()

---
## Section 5 — Effect on direct accuracy

Watching for collateral damage: an intervention that destroys task accuracy is steering the model's *answer* (or just breaking it), not its metacognition.

In [ ]:
# Section 5.1  Direct accuracy vs steering multiplier.
for run in RUNS:
    dts = _direction_types(run)
    fig, axes = plt.subplots(1, len(dts), figsize=(6.5 * len(dts), 4.2), squeeze=False)
    base_val = _baseline_value(run, 'direct_accuracy')
    for j, dt in enumerate(dts):
        _plot_steering_panel(axes[0, j], run, dt, 'direct_accuracy',
                              baseline=base_val, ylabel='direct accuracy')
        axes[0, j].set_ylim(0, 1)
    plt.tight_layout(); plt.show()

---
## Section 6 — Layer × multiplier heatmaps

Compact view of the full sweep, one heatmap per (run × direction × outcome). Cells show the *delta* from the baseline value at the same outcome. Diverging colormap → blue = decrease, red = increase.

In [ ]:
# Section 6.1  Heatmap helper: layers (rows) × multipliers (cols) of Δ-from-baseline.
def _heatmap(ax, run: RunData, direction_type: str, value_col: str, *, title: str):
    sub = run.df[(run.df['kind'] == 'steer') & (run.df['direction_type'] == direction_type)]
    if len(sub) == 0:
        ax.text(0.5, 0.5, '(no data)', ha='center', va='center'); ax.set_axis_off(); return
    layers = sorted(sub['layer'].unique().astype(int))
    mults = sorted(sub['multiplier'].unique().astype(float))
    base = _baseline_value(run, value_col)
    M = np.full((len(layers), len(mults)), np.nan)
    for _, row in sub.iterrows():
        i = layers.index(int(row['layer']))
        j = mults.index(float(row['multiplier']))
        M[i, j] = float(row[value_col]) - (base if base is not None else 0.0)
    vmax = np.nanmax(np.abs(M)) if np.any(~np.isnan(M)) else 1.0
    if vmax == 0 or not np.isfinite(vmax):
        vmax = 1.0
    im = ax.imshow(M, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
    ax.set_xticks(range(len(mults))); ax.set_xticklabels([f'{m:+g}' for m in mults], fontsize=7)
    ax.set_yticks(range(len(layers))); ax.set_yticklabels([f'L{l}' for l in layers], fontsize=7)
    ax.set_xlabel('multiplier'); ax.set_ylabel('layer')
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

OUTCOMES = [
    ('stated_confidence_mean',  'Δ stated conf'),
    ('direct_entropy_mean',     'Δ direct H'),
    ('spearman_stated_vs_direct_entropy', 'Δ Spearman ρ'),
    ('direct_accuracy',         'Δ accuracy'),
]

for run in RUNS:
    dts = _direction_types(run)
    fig, axes = plt.subplots(len(OUTCOMES), len(dts),
                              figsize=(5.2 * len(dts), 3.0 * len(OUTCOMES)),
                              squeeze=False)
    for i, (col, label) in enumerate(OUTCOMES):
        for j, dt in enumerate(dts):
            _heatmap(axes[i, j], run, dt, col, title=f'{run.label} · steer {dt} · {label}')
    plt.tight_layout(); plt.show()

---
## Section 7 — Cross-run comparison (single layer)

When multiple runs are loaded, overlay their steering curves at one layer to see whether base / instruct / finetuned respond differently to the same intervention. Edit `COMPARE_LAYER` below.

In [ ]:
COMPARE_LAYER = None  # int, e.g. 18; None → use the median layer present in the first run.

if len(RUNS) >= 2:
    if COMPARE_LAYER is None:
        layers = sorted(RUNS[0].df['layer'].dropna().unique())
        COMPARE_LAYER = int(layers[len(layers) // 2]) if layers else None
    print(f'Comparing at layer L{COMPARE_LAYER}')

    dts = _direction_types(RUNS[0])
    outcomes = [
        ('stated_confidence_mean', 'mean stated confidence'),
        ('direct_entropy_mean',    'mean direct entropy'),
        ('spearman_stated_vs_direct_entropy', 'Spearman ρ'),
    ]
    fig, axes = plt.subplots(len(outcomes), len(dts),
                              figsize=(6.5 * len(dts), 3.4 * len(outcomes)),
                              squeeze=False)
    for i, (col, ylab) in enumerate(outcomes):
        for j, dt in enumerate(dts):
            ax = axes[i, j]
            for run in RUNS:
                sub = run.df[(run.df['kind'] == 'steer') &
                              (run.df['direction_type'] == dt) &
                              (run.df['layer'] == COMPARE_LAYER)]
                if len(sub) == 0:
                    continue
                sub = sub.sort_values('multiplier')
                ax.plot(sub['multiplier'], sub[col], marker='o', linewidth=1.5,
                        markersize=4, label=run.label)
                base = _baseline_value(run, col)
                if base is not None:
                    ax.axhline(base, linestyle='--', linewidth=0.8, alpha=0.4,
                                color=ax.lines[-1].get_color())
            ax.axvline(0, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
            ax.set_xlabel('steering multiplier')
            ax.set_ylabel(ylab)
            ax.set_title(f'L{COMPARE_LAYER} · steer {dt}', fontsize=9)
            ax.grid(alpha=0.3); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()
else:
    print('Only one run loaded — load ≥2 runs in RUN_DIRS to use this section.')

---
## Section 8 — Per-question scatter (drill-down)

Pick one steering condition and plot the full per-question scatter of (direct entropy, stated confidence) against the baseline. Useful for spotting whether ρ collapse comes from a uniform shift, increased noise, or specific outliers.

In [ ]:
# Edit these to drill into a specific condition.
DRILL_RUN_LABEL: Optional[str] = None       # None → first run
DRILL_DIRECTION = 'stated_confidence'
DRILL_LAYER: Optional[int] = None             # None → median layer
DRILL_MULT: Optional[float] = None            # None → most-positive multiplier

def _condition_arrays(run: RunData, *, kind, direction_type=None, layer=None, multiplier=None):
    for cond in run.results['conditions']:
        if cond.get('kind') != kind: continue
        if direction_type is not None and cond.get('direction_type') != direction_type: continue
        if layer is not None and int(cond.get('layer', -1)) != int(layer): continue
        if multiplier is not None and float(cond.get('multiplier', np.nan)) != float(multiplier): continue
        return cond
    return None

run = next((r for r in RUNS if r.label == DRILL_RUN_LABEL), RUNS[0]) if RUNS else None
if run is None:
    print('No runs loaded.')
else:
    layers_present = sorted(run.df[(run.df['kind']=='steer') & (run.df['direction_type']==DRILL_DIRECTION)]['layer'].unique().astype(int))
    mults_present = sorted(run.df[(run.df['kind']=='steer') & (run.df['direction_type']==DRILL_DIRECTION)]['multiplier'].unique().astype(float))
    layer = DRILL_LAYER if DRILL_LAYER is not None else (layers_present[len(layers_present)//2] if layers_present else None)
    mult = DRILL_MULT if DRILL_MULT is not None else (max(mults_present) if mults_present else None)

    base = _condition_arrays(run, kind='baseline')
    cond = _condition_arrays(run, kind='steer', direction_type=DRILL_DIRECTION, layer=layer, multiplier=mult)
    if base is None or cond is None:
        print(f'Could not find baseline / steer condition (run={run.label}, dir={DRILL_DIRECTION}, L={layer}, α={mult}).')
    else:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True, sharey=True)
        for ax, c, ttl in [(axes[0], base, 'baseline'),
                            (axes[1], cond, f'steer {DRILL_DIRECTION} · L{layer} · α={mult:+g}')]:
            x = np.array(c['direct_entropy_per_question'], dtype=float)
            y = np.array(c['stated_confidence_per_question'], dtype=float)
            valid = ~np.isnan(x) & ~np.isnan(y)
            rho = float(spearmanr(x[valid], y[valid]).correlation) if valid.sum() >= 2 else float('nan')
            ax.scatter(x[valid], y[valid], s=10, alpha=0.5)
            ax.set_xlabel('direct entropy'); ax.set_ylabel('stated confidence')
            ax.set_title(f'{run.label} — {ttl}\nρ={rho:+.3f}  n={int(valid.sum())}')
            ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()